# Stage 8 — RAGAS evaluation

Input: `data/generations.jsonl` (20 questions with hybrid-retrieved context + generated answer) + `data/chunks/chunks.jsonl`.

Four RAGAS metrics, all judged by GPT-4o (wired in via Azure, since RAGAS defaults to plain OpenAI):

* **Faithfulness** — does the answer's claims actually follow from the retrieved context? (no ground truth needed)
* **Answer relevancy** — does the answer address the question asked? (uses embeddings, not ground truth)
* **Context precision** — are the relevant chunks ranked near the top of what was retrieved?
* **Context recall** — does the retrieved context contain what's needed to produce the correct answer?

## Ground truth note

RAGAS's context precision/recall need a reference answer, but `eval_set.jsonl` only has an `expected_chunk_id`.

Rather than spend more GPT-4o calls generating a polished reference answer, this notebook uses **the expected chunk's own raw content as ground truth** — it reads like a passage rather than an ideal answer, but it's sufficient for RAGAS's claim-checking and costs nothing extra.

## Cost/rate-limit warning

Each metric makes multiple LLM calls per question internally (e.g., faithfulness decomposes an answer into individual claims, then verifies each separately).

Across:

```text
4 metrics × 20 questions
```

this can result in **100+ GPT-4o calls**, not just 20.

Worth watching for `429` rate-limit errors again, same as Stage 3.

## API note

`ragas`'s exact API has changed across versions — if `evaluate(...)` errors on an unexpected argument or `ground_truth` vs `ground_truths` key mismatch, check the installed `ragas` version's documentation.

The wrapper pattern used here:

```text
LangchainLLMWrapper
LangchainEmbeddingsWrapper
```

is the most version-stable way to plug in Azure OpenAI, but exact metric names/signatures may need small tweaks depending on the installed version.


In [3]:
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from datasets import Dataset
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from ragas import evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import answer_relevancy, context_precision, context_recall, faithfulness

In [1]:
import sys
import importlib.metadata as md

print("Python:", sys.executable)

for pkg in [
    "ragas",
    "openai",
    "langchain-community",
    "langchain-openai",
]:
    try:
        print(pkg, md.version(pkg))
    except md.PackageNotFoundError:
        print(pkg, "NOT INSTALLED")

Python: /Users/rutikakadam/Documents/My Studies/Projects/multi-modal-rag/.venv-ragas/bin/python
ragas 0.3.9
openai 1.109.1
langchain-community 0.2.19
langchain-openai 0.1.25


In [2]:
from ragas import evaluate
print("RAGAS import successful")

/Users/rutikakadam/Documents/My Studies/Projects/multi-modal-rag/.venv-ragas/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RAGAS import successful


In [4]:
load_dotenv()

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
CHUNKS_PATH = DATA_DIR / "chunks" / "chunks.jsonl"
GENERATIONS_PATH = DATA_DIR / "generations.jsonl"
LOGS_DIR = DATA_DIR / "logs"
LOGS_DIR.mkdir(parents=True, exist_ok=True)

AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_API_KEY = os.environ["AZURE_OPENAI_API_KEY"]
AZURE_OPENAI_API_VERSION = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-08-01-preview")
AZURE_OPENAI_GPT4O_DEPLOYMENT = os.environ["AZURE_OPENAI_GPT4O_DEPLOYMENT"]
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"]

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    all_chunks = [json.loads(line) for line in f]
chunks_by_id = {c["chunk_id"]: c for c in all_chunks}

with open(GENERATIONS_PATH, "r", encoding="utf-8") as f:
    generations = [json.loads(line) for line in f]

print(f"loaded {len(all_chunks)} chunks, {len(generations)} generations")

loaded 1301 chunks, 20 generations


## Build the RAGAS-format records

`generations.jsonl` already carries `expected_chunk_id` (saved back in Stage 7) and `context_chunk_ids` — both just need resolving back to actual text via `chunks_by_id`.


In [5]:
def build_ragas_record(gen: dict) -> dict:
    contexts = [chunks_by_id[cid]["content"] for cid in gen["context_chunk_ids"]]
    ground_truth = chunks_by_id[gen["expected_chunk_id"]]["content"]
    return {
        "question": gen["question"],
        "answer": gen["answer"],
        "contexts": contexts,
        "ground_truth": ground_truth,
    }


ragas_records = [build_ragas_record(g) for g in generations]
pd.DataFrame(ragas_records)[["question", "answer", "ground_truth"]].head(3)

,question,answer,ground_truth
0,What demonstrates more variation in estimates ...,The upper bounds of ECS estimates demonstrate ...,There is more variation in the upper bounds of...
1,What advantage does the radiative kernel appro...,The radiative kernel approach is advantageous ...,The response to land surface temperature chang...
2,What factors contribute to permafrost degradat...,Factors contributing to permafrost degradation...,"et al., 2019). A lag between current climate c..."


## Assemble the RAGAS dataset

RAGAS consumes a Hugging Face `datasets.Dataset`, not a plain list of dictionaries.


In [6]:
ragas_dataset = Dataset.from_dict({
    "question": [r["question"] for r in ragas_records],
    "answer": [r["answer"] for r in ragas_records],
    "contexts": [r["contexts"] for r in ragas_records],
    "ground_truth": [r["ground_truth"] for r in ragas_records],
})

ragas_dataset

Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 20
})

## Wire Azure OpenAI into RAGAS

RAGAS defaults to plain OpenAI. `LangchainLLMWrapper` / `LangchainEmbeddingsWrapper` let it use any LangChain-compatible model instead — the same `AzureChatOpenAI` / `AzureOpenAIEmbeddings` classes and credentials used throughout this project, just wrapped for RAGAS's interface.


In [7]:
azure_llm = LangchainLLMWrapper(AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    azure_deployment=AZURE_OPENAI_GPT4O_DEPLOYMENT,
))

azure_embeddings = LangchainEmbeddingsWrapper(AzureOpenAIEmbeddings(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    azure_deployment=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
))

RAGAS_METRICS = [faithfulness, answer_relevancy, context_precision, context_recall]

/var/folders/9p/gm2n68xn7n1bhds792y57yg40000gn/T/ipykernel_66786/2834549709.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  azure_llm = LangchainLLMWrapper(AzureChatOpenAI(
/var/folders/9p/gm2n68xn7n1bhds792y57yg40000gn/T/ipykernel_66786/2834549709.py:8: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  azure_embeddings = LangchainEmbeddingsWrapper(AzureOpenAIEmbeddings(


### Sanity check on 2 questions before running all 20

Given the multiplied-out call volume (`4 metrics × multiple internal LLM calls each`), validate on a small slice first — same reasoning as every other stage's sanity-check cell.


In [8]:
sanity_subset = ragas_dataset.select(range(2))

sanity_result = evaluate(
    sanity_subset,
    metrics=RAGAS_METRICS,
    llm=azure_llm,
    embeddings=azure_embeddings,
)

sanity_result.to_pandas()

Evaluating: 100%|██████████| 8/8 [00:12<00:00,  1.53s/it]


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What demonstrates more variation in estimates ...,[There is more variation in the upper bounds o...,The upper bounds of ECS estimates demonstrate ...,There is more variation in the upper bounds of...,0.5,0.852777,1.0,0.5
1,What advantage does the radiative kernel appro...,[The response to land surface temperature chan...,The radiative kernel approach is advantageous ...,The response to land surface temperature chang...,1.0,0.929854,1.0,0.9


#### Run the full evaluation (all 20 questions)"

In [9]:
ragas_result = evaluate(
    ragas_dataset,
    metrics=RAGAS_METRICS,
    llm=azure_llm,
    embeddings=azure_embeddings,
)

ragas_df = ragas_result.to_pandas()
ragas_df.head(10)

Evaluating:   1%|▏         | 1/80 [00:01<02:29,  1.89s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 80/80 [03:00<00:00,  2.26s/it]


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What demonstrates more variation in estimates ...,[There is more variation in the upper bounds o...,The upper bounds of ECS estimates demonstrate ...,There is more variation in the upper bounds of...,0.5,0.852800,1.000000,0.500000
1,What advantage does the radiative kernel appro...,[The response to land surface temperature chan...,The radiative kernel approach is advantageous ...,The response to land surface temperature chang...,1.0,0.929857,1.000000,0.900000
2,What factors contribute to permafrost degradat...,"[et al., 2019). A lag between current climate ...",Factors contributing to permafrost degradation...,"et al., 2019). A lag between current climate c...",1.0,0.986770,1.000000,0.642857
3,What is the title of the study published by Po...,"[Ponater, 2019: Implication of strongly increa...",The title of the study published by Ponater in...,"Ponater, 2019: Implication of strongly increas...",1.0,0.720710,1.000000,0.142857
4,What is the title and publication year of the ...,"[Kaplan, 2020: The Relationship between the Ma...",The title of the study is 'Observational Evide...,"Hallberg, 2013: Northern High-Latitude Heat Bu...",1.0,0.478217,0.805556,0.100000
5,How do aerosols released by human activities i...,[Clouds play a critical role in the Earth's en...,Aerosols released by human activities influenc...,Clouds play a critical role in the Earth's ene...,1.0,0.993419,1.000000,1.000000
6,What evidence suggests that the true equilibri...,[(a) Equilibrium climate sensitivity estimates...,The evidence suggesting that the true equilibr...,The traditional energy budget framework lacks ...,1.0,0.992134,NaN,1.000000
7,What is the journal title and volume number in...,"[Diffenbaugh, and Singh et al., 2014: Observed...",The study by Diffenbaugh and Singh in 2014 on ...,"Diffenbaugh, and Singh et al., 2014: Observed ...",1.0,0.880076,1.000000,0.166667
8,What effect could freshwater input from meltin...,[the emergence of SH polar amplification remai...,Freshwater input from melting ice shelves coul...,the emergence of SH polar amplification remain...,1.0,0.000000,1.000000,1.000000
9,What study analyzed the amplification of the N...,"[Seager, 2013: The response of the North Ameri...","The study by Cook, B.I., R.L. Miller, and R. S...","Seager, 2013: The response of the North Americ...",1.0,0.410897,0.866667,0.333333


## Save + aggregate

Saved via pandas directly (`to_json(..., lines=True)`) rather than the usual hand-rolled `save_jsonl` helper, since `ragas_df` contains a `contexts` column of lists, which pandas' own JSON writer handles natively.


In [10]:
ragas_df.to_json(LOGS_DIR / "stage8_ragas_scores.jsonl", orient="records", lines=True)
print(f"saved {len(ragas_df)} RAGAS-scored rows to {LOGS_DIR / 'stage8_ragas_scores.jsonl'}")

METRIC_COLS = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
summary = ragas_df[METRIC_COLS].mean()
display(summary)

saved 20 RAGAS-scored rows to /Users/rutikakadam/Documents/My Studies/Projects/multi-modal-rag/data/logs/stage8_ragas_scores.jsonl


faithfulness         0.817262
answer_relevancy     0.819955
context_precision    0.934722
context_recall       0.739286
dtype: float64

## Diagnose the weakest questions per metric

Same spirit as Stages 5/6's miss-diagnosis — the aggregate average tells you **if** there's a problem, not **why**.

A low `faithfulness` row likely means the generated answer included claims that were not actually supported by the retrieved context, indicating possible hallucination.

A low `context_recall` row likely means the retrieved chunks were missing information required by the ground truth.


In [14]:
for metric in METRIC_COLS:
    print(f"--- Lowest {metric} ---")
    print(ragas_df.nsmallest(3, metric)[["user_input", metric]].to_string(index=False))
    print()


--- Lowest faithfulness ---
                                                                                                                                                      user_input  faithfulness
                                                                                        What is the likely range for the transient climate response (TCR) in °C?      0.250000
What does the slope of the line connecting energy imbalance and temperature change represent in the context of Earth's energy imbalance and climate sensitivity?      0.428571
                                                                     What demonstrates more variation in estimates of ECS, the upper bounds or the lower bounds?      0.500000

--- Lowest answer_relevancy ---
                                                                                                                                                          user_input  answer_relevancy
                                                   What 

## Pipeline complete through Stage 8

```text
Extraction
    ↓
Cleaning
    ↓
Semantic Chunking + Embeddings
    ↓
Milvus (3 Indexes)
    ↓
Retrieval + Latency/Accuracy Benchmarking
    ↓
Reranking (Baseline / MMR / Hybrid)
    ↓
Generation
    ↓
RAGAS Evaluation
```

**One thing worth flagging:** the original assignment explicitly required rendering output to a DOCX file (`"render that output over the DOCx"`) — that's Stage 9, not yet built.

Confirm whether you want that built too, since calling Stage 8 **"the last stage"** would leave that specific requirement unmet.
